In [1]:
import pandas as pd
import numpy as np
from utils import get_data

In [10]:
df = get_data("../data/XAUUSD_M5_01-01-2025_to_24-08-2026.csv")

In [11]:
def find_pivots(df, hl, hr, ll, lr):
    # Previous highs/lows
    left_high = (
        df["High"]
        .shift(1)
        .rolling(hl)
        .max()
    )

    left_low = (
        df["Low"]
        .shift(1)
        .rolling(ll)
        .min()
    )

    # Future highs/lows
    right_high = (
        df["High"]
        .shift(-1)
        .rolling(hr)
        .max()
        .shift(-(hr - 1))
    )

    right_low = (
        df["Low"]
        .shift(-1)
        .rolling(lr)
        .min()
        .shift(-(lr - 1))
    )

    df["pivot_high"] = (
        (df["High"] > left_high) &
        (df["High"] > right_high)
    )

    df["pivot_low"] = (
        (df["Low"] < left_low) &
        (df["Low"] < right_low)
    )

    return df

In [12]:
find_pivots(df, 4, 4, 4, 4)

,Open,High,Low,Close,pivot_high,pivot_low
Datetime,,,,,,
2025-03-26 04:20:00,3018.88,3019.02,3017.69,3018.34,False,False
2025-03-26 04:25:00,3018.34,3019.64,3018.34,3018.63,False,False
2025-03-26 04:30:00,3018.59,3020.46,3018.28,3020.01,False,False
2025-03-26 04:35:00,3020.01,3021.13,3019.80,3020.34,False,False
2025-03-26 04:40:00,3020.34,3022.86,3020.05,3022.86,False,False
...,...,...,...,...,...,...
2026-08-24 14:30:00,4646.61,4652.14,4643.51,4652.14,False,False
2026-08-24 14:35:00,4651.97,4670.01,4650.81,4660.51,False,False
2026-08-24 14:40:00,4660.52,4666.49,4654.22,4658.73,False,False


In [13]:
def generate_signals(df, hr, lr, sl_pips, pip_size, tp_pips=None):
    df = df.copy()

    # Last confirmed swing high/low, forward-filled until the next one appears
    last_swing_high = df["High"].where(df["pivot_high"]).shift(hr).ffill()
    last_swing_low = df["Low"].where(df["pivot_low"]).shift(lr).ffill()

    # Breakout: close crosses above/below the last confirmed swing level
    above = df["Close"] > last_swing_high
    below = df["Close"] < last_swing_low

    # Only fire on the crossing bar, not every bar price stays beyond it
    df["buy_signal"] = above & ~above.shift(1).fillna(False)
    df["sell_signal"] = below & ~below.shift(1).fillna(False)

    # --- Stop loss: two common approaches ---

    # (a) Fixed distance in pips from entry
    df["long_sl"] = df["Close"] - sl_pips * pip_size
    df["short_sl"] = df["Close"] + sl_pips * pip_size

    # (b) Structure-based: SL beyond the swing that triggered the breakout
    #     (usually preferred for this style — SL sits below/above real support/resistance
    #      rather than an arbitrary distance)
    df["long_sl_struct"] = last_swing_low
    df["short_sl_struct"] = last_swing_high

    if tp_pips:
        df["long_tp"] = df["Close"] + tp_pips * pip_size
        df["short_tp"] = df["Close"] - tp_pips * pip_size

    return df

In [14]:
generate_signals(df, 4, 4, 20, 0.01, 40)

,Open,High,Low,Close,pivot_high,pivot_low,buy_signal,sell_signal,long_sl,short_sl,long_sl_struct,short_sl_struct,long_tp,short_tp
Datetime,,,,,,,,,,,,,,
2025-03-26 04:20:00,3018.88,3019.02,3017.69,3018.34,False,False,False,False,3018.14,3018.54,NaN,NaN,3018.74,3017.94
2025-03-26 04:25:00,3018.34,3019.64,3018.34,3018.63,False,False,False,False,3018.43,3018.83,NaN,NaN,3019.03,3018.23
2025-03-26 04:30:00,3018.59,3020.46,3018.28,3020.01,False,False,False,False,3019.81,3020.21,NaN,NaN,3020.41,3019.61
2025-03-26 04:35:00,3020.01,3021.13,3019.80,3020.34,False,False,False,False,3020.14,3020.54,NaN,NaN,3020.74,3019.94
2025-03-26 04:40:00,3020.34,3022.86,3020.05,3022.86,False,False,False,False,3022.66,3023.06,NaN,NaN,3023.26,3022.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-24 14:30:00,4646.61,4652.14,4643.51,4652.14,False,False,True,False,4651.94,4652.34,4639.12,4649.06,4652.54,4651.74
2026-08-24 14:35:00,4651.97,4670.01,4650.81,4660.51,False,False,True,False,4660.31,4660.71,4639.12,4649.06,4660.91,4660.11
2026-08-24 14:40:00,4660.52,4666.49,4654.22,4658.73,False,False,True,False,4658.53,4658.93,4639.12,4649.06,4659.13,4658.33
